## Runner

In [ ]:
!python src/datasets/healthypig/test_all_trials_canonical.py --all --only-trial 1 --best-only
!python src/datasets/healthypig/test_all_trials_canonical.py --all --only-trial 2 --best-only
!python src/datasets/healthypig/test_all_trials_canonical.py --all --only-trial 3 --best-only

In [ ]:
!python src/datasets/healthypig/plot_flattening_subjects_flexion.py

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
processed_root = Path('data/138_HealthyPiG/processed')
curves=[]; pct_ref=None
for npz_path in processed_root.glob('SUBJ*/**/*_best_canonical_flexion_norm101.npz'):
    data=np.load(npz_path); pct=data['pct']
    if pct_ref is None: pct_ref=pct
    elif not np.allclose(pct_ref,pct): continue
    curves.append((data['hip_flexion'], data['knee_flexion'], data['ankle_dorsiflexion']))
if curves:
    hips=np.stack([c[0] for c in curves]); kns=np.stack([c[1] for c in curves]); ank=np.stack([c[2] for c in curves])
    out_npz=processed_root/'population_angles_norm101.npz'
    out_csv=processed_root/'population_angles_norm101.csv'
    np.savez(out_npz, pct=pct_ref, hip_mean=hips.mean(0)[:,None], knee_mean=kns.mean(0)[:,None], ankle_mean=ank.mean(0)[:,None], hip_std=hips.std(0)[:,None], knee_std=kns.std(0)[:,None], ankle_std=ank.std(0)[:,None])
    pd.DataFrame({'pct': pct_ref, 'hip_mean': hips.mean(0), 'hip_std': hips.std(0), 'knee_mean': kns.mean(0), 'knee_std': kns.std(0), 'ankle_mean': ank.mean(0), 'ankle_std': ank.std(0)}).to_csv(out_csv, index=False)
    print('Rebuilt canonical', out_npz)